# Experiment 1 — Video-level aggregation for UCF101 pose recognition

**Purpose:** evaluate the existing best BiLSTM + temporal-attention checkpoint by averaging its window-level class probabilities for each source video. This does **not** train a model and does not overwrite any checkpoint.

Your current model's reported window-level test accuracy was about 68.5%. Video-level accuracy is a different metric, so compare it separately rather than treating it as a direct like-for-like replacement.

Run these cells in order. If your project is stored somewhere else, edit `PROJECT_DIR` in Cell 2.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
SEQUENCE_DIR = PROJECT_DIR / "data" / "processed" / "sequences_all"
MODEL_PATH = PROJECT_DIR / "outputs" / "models" / "lstm_bilstm_attention_best.pt"
REPORT_DIR = PROJECT_DIR / "outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 0

print("Project:", PROJECT_DIR)
print("Device:", DEVICE)
print("Checkpoint exists:", MODEL_PATH.is_file())
assert SEQUENCE_DIR.is_dir(), f"Sequence folder not found: {SEQUENCE_DIR}"
assert MODEL_PATH.is_file(), f"Checkpoint not found: {MODEL_PATH}"


Project: C:\Users\admin\ViTPose_Action_Recognition
Device: cuda
Checkpoint exists: True


## 1. Load the test arrays

The notebook expects the existing `X_test.npy`, `y_test.npy`, and `videos_test.npy` files in `data/processed/sequences_all/`. The video ID for each window is used to group predictions back to the source video.

In [2]:
X_test = np.load(SEQUENCE_DIR / "X_test.npy", mmap_mode="r")
y_test = np.load(SEQUENCE_DIR / "y_test.npy", mmap_mode="r")
videos_test = np.load(SEQUENCE_DIR / "videos_test.npy", mmap_mode="r", allow_pickle=False)

assert X_test.ndim == 3, f"Expected [windows, time, features], got {X_test.shape}"
assert len(X_test) == len(y_test) == len(videos_test), (
    f"Length mismatch: X={len(X_test)}, y={len(y_test)}, videos={len(videos_test)}"
)

NUM_CLASSES = int(np.max(np.asarray(y_test, dtype=np.int64))) + 1
print("Test windows:", len(X_test))
print("Window shape:", X_test.shape)
print("Unique source videos:", len(np.unique(videos_test)))
print("Number of classes inferred from test labels:", NUM_CLASSES)


Test windows: 21009
Window shape: (21009, 16, 51)
Unique source videos: 2441
Number of classes inferred from test labels: 101


## 2. Rebuild the same model architecture and load the saved checkpoint

This matches the `PoseLSTM` architecture used in `Improved_UCF101_BiLSTM_Attention.ipynb`. The checkpoint is loaded for inference only.

In [3]:
class PoseLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, num_classes),
        )

    def forward(self, x):
        x = self.input_norm(x)
        sequence_output, _ = self.lstm(x)
        scores = self.attention(sequence_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(sequence_output * weights.unsqueeze(-1), dim=1)
        return self.classifier(context)


# weights_only=False supports checkpoints that include ordinary Python metadata.
# Only load checkpoints you trust (this is your locally saved project checkpoint).
try:
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
except TypeError:  # compatibility with older PyTorch versions
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = PoseLSTM(
    input_size=int(checkpoint["input_size"]),
    hidden_size=int(checkpoint["hidden_size"]),
    num_layers=int(checkpoint["num_layers"]),
    num_classes=int(checkpoint["num_classes"]),
    dropout=float(checkpoint["dropout"]),
).to(DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

assert X_test.shape[2] == int(checkpoint["input_size"]), (
    f"Feature mismatch: test has {X_test.shape[2]} features, "
    f"checkpoint expects {checkpoint['input_size']}"
)
print("Loaded checkpoint:", MODEL_PATH.name)
print("Best epoch:", checkpoint.get("best_epoch", "not stored"))
print("Checkpoint classes:", checkpoint["num_classes"])


Loaded checkpoint: lstm_bilstm_attention_best.pt
Best epoch: 2
Checkpoint classes: 101


## 3. Predict every test window

The model is used in evaluation mode with gradients disabled. Softmax probabilities are saved for aggregation.

In [4]:
class TestWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = np.array(self.X[idx], dtype=np.float32, copy=True)
        y = int(self.y[idx])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


test_loader = DataLoader(
    TestWindowDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

all_probs = []
all_window_labels = []
with torch.inference_mode():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        logits = model(xb)
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_window_labels.append(yb.numpy())

window_probs = np.concatenate(all_probs, axis=0)
window_labels = np.concatenate(all_window_labels, axis=0).astype(np.int64)
video_ids = np.asarray(videos_test)

assert len(window_probs) == len(window_labels) == len(video_ids)
print("Predicted windows:", len(window_probs))
print("Probability array:", window_probs.shape)


Predicted windows: 21009
Probability array: (21009, 101)


## 4. Average probabilities by source video

For each source video, this averages the class probabilities of all windows belonging to it, then selects the class with the highest mean probability. It also verifies that every window assigned to a video has the same ground-truth label.

In [5]:
unique_videos, inverse = np.unique(video_ids, return_inverse=True)
n_videos = len(unique_videos)
num_classes = window_probs.shape[1]

prob_sums = np.zeros((n_videos, num_classes), dtype=np.float64)
np.add.at(prob_sums, inverse, window_probs)
window_counts = np.bincount(inverse, minlength=n_videos)
video_probs = prob_sums / window_counts[:, None]

video_labels = np.empty(n_videos, dtype=np.int64)
for i in range(n_videos):
    labels_for_video = np.unique(window_labels[inverse == i])
    if len(labels_for_video) != 1:
        raise ValueError(
            f"Video {unique_videos[i]!r} has inconsistent labels: {labels_for_video}"
        )
    video_labels[i] = labels_for_video[0]

video_predictions = video_probs.argmax(axis=1)

metrics = {
    "checkpoint": MODEL_PATH.name,
    "best_epoch": int(checkpoint["best_epoch"]) if "best_epoch" in checkpoint else None,
    "test_windows": int(len(window_labels)),
    "test_videos": int(n_videos),
    "window_level_accuracy": float(accuracy_score(window_labels, window_probs.argmax(axis=1))),
    "video_level_accuracy": float(accuracy_score(video_labels, video_predictions)),
    "video_level_balanced_accuracy": float(balanced_accuracy_score(video_labels, video_predictions)),
    "video_level_macro_f1": float(f1_score(
        video_labels, video_predictions,
        labels=np.arange(num_classes), average="macro", zero_division=0
    )),
}
print(json.dumps(metrics, indent=2))


{
  "checkpoint": "lstm_bilstm_attention_best.pt",
  "best_epoch": 2,
  "test_windows": 21009,
  "test_videos": 2441,
  "window_level_accuracy": 0.6854681327050312,
  "video_level_accuracy": 0.7017615731257681,
  "video_level_balanced_accuracy": 0.6720618777233774,
  "video_level_macro_f1": 0.6704823974251416
}


## 5. Save the metrics and per-video predictions

Outputs are written to `outputs/reports/`. This creates new report files only; it does not alter the model or dataset.

In [6]:
metrics_path = REPORT_DIR / "video_level_aggregation_metrics.json"
predictions_path = REPORT_DIR / "video_level_aggregation_predictions.csv"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

per_video = pd.DataFrame({
    "video_id": unique_videos.astype(str),
    "true_label": video_labels,
    "predicted_label": video_predictions,
    "number_of_windows": window_counts,
    "correct": video_labels == video_predictions,
})
per_video.to_csv(predictions_path, index=False)

print("Saved metrics:", metrics_path)
print("Saved per-video predictions:", predictions_path)
display(per_video.head(10))


Saved metrics: C:\Users\admin\ViTPose_Action_Recognition\outputs\reports\video_level_aggregation_metrics.json
Saved per-video predictions: C:\Users\admin\ViTPose_Action_Recognition\outputs\reports\video_level_aggregation_predictions.csv


,video_id,true_label,predicted_label,number_of_windows,correct
0,116,0,59,5,False
1,117,0,0,13,True
2,118,0,0,4,True
3,119,0,1,5,False
4,120,0,0,13,True
5,121,0,0,6,True
6,122,0,77,5,False
7,123,0,77,5,False
8,124,0,77,10,False
9,125,0,17,7,False


## Important interpretation

- **Video-level accuracy** is the primary result of this experiment.
- The previous ~68.5% result was measured per window, so the two numbers are not directly comparable.
- This experiment uses the held-out test split only for reporting. Avoid changing model settings based on repeated test-set experiments; use validation results for model selection.
- If video-level accuracy remains below 75%, the next step is a new experiment with motion features or a stronger temporal model—not repeatedly tuning this test result.